# Δημιουργία Επαναχρησιμοποιήσιμης Βιβλιοθήκης Αναλογιστικής Τιμολόγησης με την PROC FCMP

## Σύνοψη για τη Διοίκηση

Μια ασφαλιστική εταιρεία κλάδων ζημιών ενσωματώνει τα βασικά μαθηματικά τιμολόγησής της — καθαρό ασφάλιστρο, φόρτιση εξόδων/κινδύνου, συνδυασμό αξιοπιστίας περιορισμένης διακύμανσης και εκτίμηση προεξοφλημένου αποθέματος — ως προσαρμοσμένες συναρτήσεις και μια υπορουτίνα πολλαπλής εξόδου στην **PROC FCMP**. Η μεταγλωττισμένη βιβλιοθήκη καταχωρείται μέσω της επιλογής συστήματος `CMPLIB=` και στη συνέχεια καλείται γραμμή προς γραμμή από ένα βήμα DATA που τιμολογεί ένα συνθετικό χαρτοφυλάκιο 100 συμβολαίων. Κάθε τιμολογημένο μέγεθος σε αυτό το σημειωματάριο — ο συντελεστής αξιοπιστίας `z`, το συνδυασμένο καθαρό ασφάλιστρο, το χρεωθέν ασφάλιστρο και η παρούσα αξία του αποθέματος υπόθεσης — παράγεται από αυτές τις μεταγλωττισμένες ρουτίνες, όχι από ενσωματωμένη αριθμητική. Το αποτέλεσμα: ο υπονοούμενος δείκτης ζημιών διαμορφώνεται στο 60.5% (αγροτική), 55.8% (προαστιακή) και 63.6% (αστική) — άνετα κάτω από το 100% σε κάθε τμήμα, επιβεβαιώνοντας ότι το φορτισμένο ασφάλιστρο καλύπτει την αναμενόμενη ζημία ενώ το βήμα τιμολόγησης παραμένει καθαρό και ελέγξιμο.

## Πηγές Δεδομένων

| Σύνολο Δεδομένων | Γραμμές | Περιγραφή | Βασικές Μεταβλητές |
|---------|------|-------------|---------------|
| `work.portfolio` | 100 | Συνθετικό ενεργό χαρτοφυλάκιο συμβολαίων κλάδων ζημιών, παραγόμενο εν σειρά με `rand()` | `policy_id`, `region` (αστική/προαστιακή/αγροτική), `years_insured`, `exposure` (έτη-οχήματος), `claim_count` (Poisson), `incurred_loss` (σφοδρότητα gamma x πλήθος), `class_pure_prem` (χειροκίνητος συντελεστής ανά περιοχή) |

Το βήμα DATA επαναλαμβάνεται σε ένα ευρύτερο εύρος `policy_id`, αλλά αυτό το περιβάλλον εκτελείται χωρίς άδεια χρήσης, οπότε η έξοδος περιορίζεται στις πρώτες **100 παρατηρήσεις** — το τιμολογημένο βιβλίο παρακάτω αντικατοπτρίζει αυτά τα 100 συμβόλαια.

# Δημιουργία Επαναχρησιμοποιήσιμης Βιβλιοθήκης Αναλογιστικής Τιμολόγησης με την PROC FCMP

Οι αναλογιστές επαναλαμβάνουν τους ίδιους υπολογισμούς σε εργασίες τιμολόγησης, αποθεματοποίησης και αναφοράς: μετατρέπουν τις ζημίες σε *καθαρό ασφάλιστρο*, εφαρμόζουν *φορτίσεις εξόδων και κινδύνου* για να φτάσουν σε ένα χρεωθέν ασφάλιστρο, συνδυάζουν την ίδια εμπειρία ενός συμβολαίου με έναν συντελεστή κατηγορίας χρησιμοποιώντας *αξιοπιστία*, και *προεξοφλούν* μελλοντικές ταμειακές ροές σε παρούσα αξία. Η επανάληψη αυτών των τύπων σε κάθε βήμα DATA προκαλεί λάθη αντιγραφής-επικόλλησης και κάνει τις αλλαγές παραδοχών επίπονες.

Η **PROC FCMP** (ο μεταγλωττιστής συναρτήσεων του SAS) μας επιτρέπει να ορίσουμε κάθε τύπο μία φορά ως ονομασμένη συνάρτηση ή υπορουτίνα, να αποθηκεύσουμε τις μεταγλωττισμένες ρουτίνες σε μια βιβλιοθήκη και έπειτα να τις καλούμε όπως κάθε ενσωματωμένη συνάρτηση SAS. Σε αυτό το σημειωματάριο:

1. Μεταγλωττίζουμε μια μικρή βιβλιοθήκη αναλογιστικών συναρτήσεων με την `PROC FCMP`.
2. Την καταχωρούμε για τη συνεδρία με την επιλογή συστήματος `CMPLIB=`.
3. Δημιουργούμε ένα συνθετικό χαρτοφυλάκιο κλάδων ζημιών.
4. Τιμολογούμε κάθε συμβόλαιο καλώντας τις προσαρμοσμένες συναρτήσεις μας και μια υπορουτίνα πολλαπλής εξόδου από ένα ενιαίο βήμα DATA.
5. Συνοψίζουμε και ερμηνεύουμε το τιμολογημένο χαρτοφυλάκιο.

## Βήμα 1 — Δημιουργία συνθετικού χαρτοφυλακίου συμβολαίων

Προσομοιώνουμε ένα βιβλίο ενεργών συμβολαίων αυτοκινήτου (τα πρώτα 100 τιμολογούνται παρακάτω υπό το όριο του περιβάλλοντος χωρίς άδεια χρήσης). Κάθε συμβόλαιο ανήκει σε μια περιοχή τιμολόγησης με το δικό της χειροκίνητο *καθαρό ασφάλιστρο* (αναμενόμενη ζημία ανά έτος-όχημα). Ο αριθμός αξιώσεων ακολουθεί μια διαδικασία Poisson κλιμακούμενη κατά την έκθεση, και οι σφοδρότητες ακολουθούν κατανομή gamma, οπότε η `incurred_loss` είναι μια ρεαλιστική σύνθετη (Poisson x gamma) ζημία. Το `years_insured` θα καθορίσει αργότερα τον συντελεστή βαρύτητας αξιοπιστίας. Ένας σταθερός σπόρος μέσω της `call streaminit` καθιστά τα δεδομένα αναπαραγώγιμα.

In [1]:
ΔΕΔΟΜΕΝΑ portfolio;
    CALL streaminit(20260531);
    LENGTH region $30;
    ΕΠΑΝΑΛΗΨΗ policy_id = 10001 ΕΩΣ 12000;
        /* Ανάθεση περιοχής τιμολόγησης */
        u = rand('uniform');
        ΕΑΝ u < 0.40 ΤΟΤΕ ΕΠΑΝΑΛΗΨΗ; region = 'αστική';    base_pp = 820; lambda = 0.18; ΤΕΛΟΣ;
        ΑΛΛΙΩΣ ΕΑΝ u < 0.75 ΤΟΤΕ ΕΠΑΝΑΛΗΨΗ; region = 'προαστιακή'; base_pp = 540; lambda = 0.11; ΤΕΛΟΣ;
        ΑΛΛΙΩΣ ΕΠΑΝΑΛΗΨΗ; region = 'αγροτική';    base_pp = 360; lambda = 0.07; ΤΕΛΟΣ;

        /* Διάρκεια ασφάλισης (έτη) και έκθεση (έτη-οχήματος) */
        years_insured = 1 + rand('poisson', 3);
        exposure = round(0.5 + rand('gamma', 4) / 4, 0.01);

        /* Σύνθετη διαδικασία αξιώσεων: συχνότητα Poisson, σφοδρότητα gamma */
        claim_count = rand('poisson', lambda * exposure);
        incurred_loss = 0;
        ΕΠΑΝΑΛΗΨΗ c = 1 ΕΩΣ claim_count;
            incurred_loss = incurred_loss + rand('gamma', 2.0) * 2500;
        ΤΕΛΟΣ;
        incurred_loss = round(incurred_loss, 0.01);

        /* Χειροκίνητο καθαρό ασφάλιστρο κατηγορίας για την έκθεση αυτού του συμβολαίου */
        class_pure_prem = round(base_pp * exposure, 0.01);

        ΕΞΟΔΟΣ;
    ΤΕΛΟΣ;
    ΚΡΑΤΗΣΗ policy_id region years_insured exposure claim_count
         incurred_loss class_pure_prem;
ΕΚΤΕΛΕΣΗ;

ΔΙΑΔΙΚΑΣΙΑ ΕΚΤΥΠΩΣΗ ΔΕΔΟΜΕΝΑ=portfolio(obs=8) noobs ΕΤΙΚΕΤΑ;
    TITLE "Πρώτα 8 Προσομοιωμένα Συμβόλαια";
    ΕΤΙΚΕΤΑ policy_id="Κωδικός Συμβολαίου" region="Περιοχή" years_insured="Έτη Ασφάλισης"
          exposure="Έκθεση (έτη-οχήματος)" claim_count="Αριθμός Αξιώσεων"
          incurred_loss="Επελθούσα Ζημία" class_pure_prem="Καθαρό Ασφάλιστρο Κατηγορίας";
ΕΚΤΕΛΕΣΗ;

                                            Πρώτα 8 Προσομοιωμένα Συμβόλαια                                             

                 Κωδικός Συμβολαίου               Περιοχή              Έτη Ασφάλισης                   Έκθεση (έτη-οχήματος)                 Αριθμός Αξιώσεων                Επελθούσα Ζημία                            Καθαρό Ασφάλιστρο Κατηγορίας
                              10001  αγροτική                                      5                                    1.36                                0                              0                                                   489.6
                              10002  αστική                                        8                                    0.97                                1                        3432.28                                                   795.4
                              10003  αστική                                        2                                    1.53                   


NOTE: DATA portfolio

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote portfolio (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.42 seconds
  cpu   0.42 seconds
NOTE: PROC PRINT data=portfolio

NOTE: PROC PRINT completed: 8 observations printed, 7 variables


## Βήμα 2 — Μεταγλώττιση της βιβλιοθήκης αναλογιστικών συναρτήσεων

Τώρα η καρδιά του σημειωματάριου. Η `PROC FCMP` με `OUTLIB=work.actfuncs.pricing` μεταγλωττίζει τέσσερις συναρτήσεις και μία υπορουτίνα στο πακέτο `pricing` του συνόλου δεδομένων `work.actfuncs`:

- **`pure_premium`** — παρατηρούμενη ζημία ανά μονάδα έκθεσης (συχνότητα x σφοδρότητα συνδυασμένες).
- **`credibility_z`** — συντελεστής αξιοπιστίας περιορισμένης διακύμανσης `Z = sqrt(n / (n + k))`, όπου `n` είναι τα έτη έκθεσης του συμβολαίου και `k` μια σταθερά ρύθμισης.
- **`charged_premium`** — εφαρμόζει μια αναλογική φόρτιση κινδύνου και έναν σταθερό συντελεστή εξόδων σε ένα κόστος ζημίας για να φτάσει στο ασφάλιστρο που χρεώνουμε πραγματικά.
- **`pv_reserve`** — παρούσα αξία μιας μελλοντικής πληρωμής αξίωσης, `FV / (1+r)**t`, που χρησιμοποιείται για την προεξόφληση αποθεμάτων υπόθεσης.
- **`blend_premium`** (μια ΥΠΟΡΟΥΤΙΝΑ) — χρησιμοποιεί το `OUTARGS` για να επιστρέψει *δύο* τιμές ταυτόχρονα: το σταθμισμένο κατά αξιοπιστία καθαρό ασφάλιστρο και τον συντελεστή αξιοπιστίας που χρησιμοποίησε, ώστε το βήμα DATA να λαμβάνει και τα δύο σε μία μόνο κλήση.

Κάθε ρουτίνα κλείνει με `ENDSUB`, και η υπορουτίνα ονομάζει τα εγγράψιμα ορίσματά της με `OUTARGS`.

In [2]:
ΔΙΑΔΙΚΑΣΙΑ fcmp outlib=work.actfuncs.pricing;

    /* Παρατηρούμενο καθαρό ασφάλιστρο: κόστος ζημίας ανά μονάδα έκθεσης */
    function pure_premium(loss, exposure);
        ΕΑΝ exposure <= 0 ΤΟΤΕ RETURN(.);
        RETURN(loss / exposure);
    endsub;

    /* Αξιοπιστία περιορισμένης διακύμανσης: Z = sqrt(n / (n + k)) */
    function credibility_z(n_years, k);
        ΕΑΝ n_years <= 0 ΤΟΤΕ RETURN(0);
        RETURN(sqrt(n_years / (n_years + k)));
    endsub;

    /* Χρεωθέν ασφάλιστρο = κόστος ζημίας προσαυξημένο για φόρτιση κινδύνου + έξοδα */
    function charged_premium(loss_cost, risk_load, expense_ratio);
        gross = loss_cost * (1 + risk_load);
        ΕΑΝ expense_ratio >= 1 ΤΟΤΕ RETURN(.);
        RETURN(gross / (1 - expense_ratio));
    endsub;

    /* Παρούσα αξία μιας μελλοντικής πληρωμής αξίωσης προεξοφλημένης t έτη με επιτόκιο r */
    function pv_reserve(future_value, r, t);
        RETURN(future_value / (1 + r) ** t);
    endsub;

    /* Συνδυασμός αξιοπιστίας: επιστρέφει το σταθμισμένο καθαρό ασφάλιστρο ΚΑΙ το Z που χρησιμοποιήθηκε */
    subroutine blend_premium(own_pp, class_pp, n_years, k, blended, z_used);
        outargs blended, z_used;
        z_used = credibility_z(n_years, k);
        blended = z_used * own_pp + (1 - z_used) * class_pp;
    endsub;

ΕΚΤΕΛΕΣΗ;


               The FCMP Procedure

  Output Library: work.actfuncs.pricing
  User-defined Function: pure_premium
  User-defined Function: credibility_z
  User-defined Function: charged_premium
  User-defined Function: pv_reserve
  User-defined Subroutine: blend_premium




NOTE: PROC FCMP outlib=work.actfuncs.pricing

NOTE: Function pure_premium stored to work.actfuncs.pricing.
NOTE: Function credibility_z stored to work.actfuncs.pricing.
NOTE: Function charged_premium stored to work.actfuncs.pricing.
NOTE: Function pv_reserve stored to work.actfuncs.pricing.
NOTE: Subroutine blend_premium stored to work.actfuncs.pricing.


## Βήμα 3 — Καταχώρηση της βιβλιοθήκης με CMPLIB=

Η μεταγλώττιση των ρουτινών δεν αρκεί· το SAS πρέπει να ενημερωθεί πού να τις βρει όταν ένα μεταγενέστερο βήμα DATA ή μια διαδικασία αναφέρεται σε ένα όνομα που δεν αναγνωρίζει ως ενσωματωμένο. Η επιλογή συστήματος `CMPLIB=` δείχνει στο σύνολο δεδομένων (όχι στο τριεπίπεδο όνομα πακέτου) που περιέχει τον μεταγλωττισμένο κώδικα. Μετά από αυτή τη δήλωση `OPTIONS`, κάθε συνάρτηση και υπορουτίνα παραπάνω είναι καλέσιμη με το όνομά της.

In [3]:
OPTIONS cmplib=work.actfuncs;


NOTE: Option CMPLIB changed to WORK.ACTFUNCS.


## Βήμα 4 — Τιμολόγηση του χαρτοφυλακίου με τις προσαρμοσμένες συναρτήσεις

Το βήμα DATA τιμολόγησης είναι πλέον σχεδόν απαλλαγμένο από αριθμητική — η αναλογιστική πρόθεση διαβάζεται απευθείας από τα ονόματα των συναρτήσεων. Για κάθε συμβόλαιο:

1. Υπολογίζουμε το δικό του παρατηρούμενο καθαρό ασφάλιστρο με την `pure_premium`.
2. Καλούμε την υπορουτίνα `blend_premium` για να το σταθμίσουμε κατά αξιοπιστία έναντι του συντελεστή κατηγορίας της περιοχής, ανακτώντας τόσο το συνδυασμένο κόστος ζημίας όσο και τον συντελεστή αξιοπιστίας `z` μέσω του `OUTARGS`.
3. Προσαυξάνουμε το συνδυασμένο κόστος ζημίας σε χρεωθέν ασφάλιστρο με φόρτιση κινδύνου 12% και συντελεστή εξόδων 25% μέσω της `charged_premium`.
4. Εκτιμούμε το ακόμη ανοιχτό απόθεμα υπόθεσης ως 35% της επελθούσας ζημίας του συμβολαίου και το προεξοφλούμε τρία έτη με επιτόκιο 4% σε παρούσα αξία με την `pv_reserve`.

Σημειώστε πώς τα ορίσματα εξόδου της υπορουτίνας (`blended_pp`, `z`) πρέπει να αρχικοποιούνται πριν από το `CALL`. Η παρούσα αξία του αποθέματος διαφέρει από συμβόλαιο σε συμβόλαιο επειδή καθορίζεται από την πραγματική επελθούσα ζημία κάθε συμβολαίου — συμβόλαια χωρίς αξιώσεις φέρουν μηδενικό απόθεμα, οπότε και το `reserve_pv` τους είναι μηδέν.

In [4]:
ΔΕΔΟΜΕΝΑ rated;
    ΟΡΙΣΜΟΣ portfolio;

    /* 1. Δική του εμπειρία ζημιών ως καθαρό ασφάλιστρο */
    own_pp = round(pure_premium(incurred_loss, exposure), 0.01);

    /* 2. Στάθμιση κατά αξιοπιστία της δικής του εμπειρίας έναντι του συντελεστή κατηγορίας.
          k = 4 έτη έκθεσης για σχεδόν πλήρη αξιοπιστία. */
    blended_pp = .;
    z = .;
    CALL blend_premium(own_pp, class_pure_prem / exposure,
                       years_insured, 4, blended_pp, z);
    blended_pp = round(blended_pp, 0.01);
    z = round(z, 0.001);

    /* 3. Μετατροπή του συνδυασμένου κόστους ζημίας (ανά έτος-όχημα) σε χρεωθέν ασφάλιστρο */
    loss_cost = blended_pp * exposure;
    premium = round(charged_premium(loss_cost, 0.12, 0.25), 0.01);

    /* 4. Εκκρεμές απόθεμα υπόθεσης = 35% της επελθούσας ζημίας, αναμένεται
          να διακανονιστεί σε 3 έτη· προεξόφλησέ το σε παρούσα αξία με 4%. */
    case_reserve = round(0.35 * incurred_loss, 0.01);
    reserve_pv   = round(pv_reserve(case_reserve, 0.04, 3), 0.01);

    ΚΡΑΤΗΣΗ policy_id region years_insured exposure incurred_loss
         own_pp class_pure_prem blended_pp z premium
         case_reserve reserve_pv;
ΕΚΤΕΛΕΣΗ;

ΔΙΑΔΙΚΑΣΙΑ ΕΚΤΥΠΩΣΗ ΔΕΔΟΜΕΝΑ=rated(obs=10) noobs ΕΤΙΚΕΤΑ;
    TITLE "Τιμολογημένο Χαρτοφυλάκιο (πρώτα 10 συμβόλαια)";
    ΜΕΤΑΒΛΗΤΗ policy_id region years_insured exposure own_pp
        blended_pp z premium reserve_pv;
    ΕΤΙΚΕΤΑ policy_id="Κωδικός Συμβολαίου" region="Περιοχή" years_insured="Έτη Ασφάλισης"
          exposure="Έκθεση (έτη-οχήματος)" own_pp="Ίδιο Καθαρό Ασφάλιστρο"
          blended_pp="Συνδυασμένο Καθαρό Ασφάλιστρο" z="Συντελεστής Αξιοπιστίας (z)"
          premium="Χρεωθέν Ασφάλιστρο" reserve_pv="Παρούσα Αξία Αποθέματος";
ΕΚΤΕΛΕΣΗ;

                                     Τιμολογημένο Χαρτοφυλάκιο (πρώτα 10 συμβόλαια)                                     

                 Κωδικός Συμβολαίου               Περιοχή              Έτη Ασφάλισης                   Έκθεση (έτη-οχήματος)                      Ίδιο Καθαρό Ασφάλιστρο                             Συνδυασμένο Καθαρό Ασφάλιστρο                        Συντελεστής Αξιοπιστίας (z)                   Χρεωθέν Ασφάλιστρο                       Παρούσα Αξία Αποθέματος
                              10001  αγροτική                                      5                                    1.36                                           0                                                     91.67                                              0.745                               186.18                                             0
                              10002  αστική                                        8                                    0.97                                


NOTE: DATA rated


NOTE: Read 100 rows from portfolio.
NOTE: Wrote rated (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.06 seconds
  cpu   0.06 seconds
NOTE: PROC PRINT data=rated

NOTE: PROC PRINT completed: 10 observations printed, 9 variables


## Βήμα 5 — Σύνοψη του τιμολογημένου βιβλίου

Με κάθε συμβόλαιο τιμολογημένο μέσω της ίδιας μεταγλωττισμένης βιβλιοθήκης, μπορούμε να συγκεντρώσουμε το χαρτοφυλάκιο ανά περιοχή. Αναφέρουμε το μέσο χρεωθέν ασφάλιστρο, τον μέσο συντελεστή αξιοπιστίας, τη συνολική επελθούσα ζημία και τη συνολική παρούσα αξία αποθέματος υπόθεσης, και έπειτα υπολογίζουμε τον υπονοούμενο δείκτη ζημιών ανά τμήμα ώστε να δούμε αν το φορτισμένο ασφάλιστρο καλύπτει την αναμενόμενη ζημία.

In [5]:
ΔΙΑΔΙΚΑΣΙΑ ΜΕΣΟΙ ΔΕΔΟΜΕΝΑ=rated mean sum maxdec=2 nonobs;
    ΚΛΑΣΗ region;
    ΜΕΤΑΒΛΗΤΗ premium z incurred_loss reserve_pv;
    ΕΤΙΚΕΤΑ region="Περιοχή" premium="Χρεωθέν Ασφάλιστρο" z="Συντελεστής Αξιοπιστίας (z)"
          incurred_loss="Επελθούσα Ζημία" reserve_pv="Παρούσα Αξία Αποθέματος";
    TITLE "Σύνοψη Χαρτοφυλακίου ανά Περιοχή Τιμολόγησης";
ΕΚΤΕΛΕΣΗ;

/* Υπονοούμενος δείκτης ζημιών = επελθούσα ζημία / χρεωθέν ασφάλιστρο, συν
   την παρούσα αξία αποθέματος που φέρει το τμήμα, ανά περιοχή. */
ΔΙΑΔΙΚΑΣΙΑ SQL;
    TITLE "Υπονοούμενος Δείκτης Ζημιών και Παρούσα Αξία Αποθέματος ανά Περιοχή";
    ΕΠΙΛΟΓΗ region ΕΤΙΚΕΤΑ="Περιοχή",
           count(*)                              AS n_policies ΕΤΙΚΕΤΑ="Αριθμός Συμβολαίων",
           sum(incurred_loss)                    AS total_loss     ΜΟΡΦΗ=dollar12.2 ΕΤΙΚΕΤΑ="Συνολική Ζημία",
           sum(premium)                          AS total_premium  ΜΟΡΦΗ=dollar12.2 ΕΤΙΚΕΤΑ="Συνολικό Ασφάλιστρο",
           sum(incurred_loss) / sum(premium)     AS loss_ratio     ΜΟΡΦΗ=percent8.1 ΕΤΙΚΕΤΑ="Δείκτης Ζημιών",
           sum(reserve_pv)                       AS total_pv_reserve ΜΟΡΦΗ=dollar12.2 ΕΤΙΚΕΤΑ="Συνολική ΠΑ Αποθέματος"
    FROM rated
    GROUP ΚΑΤΑ region
    ORDER ΚΑΤΑ region;
QUIT;
TITLE;

                                      Σύνοψη Χαρτοφυλακίου ανά Περιοχή Τιμολόγησης                                      

                                                  The MEANS Procedure

                            Analysis Variable : premium Χρεωθέν Ασφάλιστρο

        Περιοχή                        Mean            Sum
        --------------------------------------------------
        αγροτική                     476.61       11438.62
        αστική                      1987.17       67563.93
        προαστιακή                   813.04       34147.74
        --------------------------------------------------

                        Analysis Variable : z Συντελεστής Αξιοπιστίας (z)

        Περιοχή                        Mean            Sum
        --------------------------------------------------
        αγροτική                       0.71          17.14
        αστική                         0.70          23.90
        προαστιακή                     0.68          28.36
      


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC SQL 

NOTE: PROC SQL statement used.


## Ερμηνεία των Αποτελεσμάτων

Το βήμα DATA τιμολόγησης δεν γράφει ποτέ έναν μεμονωμένο τύπο προεξόφλησης ή αξιοπιστίας ενσωματωμένο απευθείας — απλώς καλεί τις `pure_premium`, `blend_premium`, `charged_premium` και `pv_reserve`. Αυτό είναι το όφελος της **PROC FCMP**: οι αναλογιστικές παραδοχές ζουν σε μία μεταγλωττισμένη βιβλιοθήκη που μπορεί να ελεγχθεί μονοδιαία, να τεθεί υπό έλεγχο εκδόσεων και να επαναχρησιμοποιηθεί σε εργασίες τιμολόγησης, αποθεματοποίησης και αναφοράς. Η αλλαγή της σταθεράς αξιοπιστίας `k`, της φόρτισης κινδύνου ή του συντελεστή εξόδων είναι μία μόνο επεξεργασία στη βιβλιοθήκη, όχι ένα κυνήγι σε κάθε πρόγραμμα.

Διαβάζοντας το τιμολογημένο βιβλίο και τη συγκεντρωτική ανά περιοχή:

- Η **Αξιοπιστία (`z`)** αυξάνεται με τα `years_insured`, ακριβώς όπως υπαγορεύει ο τύπος περιορισμένης διακύμανσης `Z = sqrt(n / (n + k))`. Ανάμεσα στα πρώτα δέκα συμβόλαια, το μονοετές συμβόλαιο προαστιακής περιοχής (10006) φτάνει `z = 0.447`, το διετές συμβόλαιο αστικής περιοχής (10003) `z = 0.577`, ενώ το εννιαετές συμβόλαιο προαστιακής περιοχής (10004) φτάνει `z = 0.832`. Τα συμβόλαια με μικρή εμπειρία έλκονται προς τον συντελεστή κατηγορίας της περιοχής· τα μακροχρόνια στηρίζονται στις δικές τους ζημίες.
- **Ο συνδυασμός σε δράση:** τα συμβόλαια χωρίς αξιώσεις (το μεγαλύτερο μέρος του βιβλίου) έχουν `own_pp = 0`, οπότε η `blend_premium` επιστρέφει ένα `blended_pp` κοντά στο `(1 - z)` επί τον συντελεστή κατηγορίας — π.χ. το συμβόλαιο 10001 (αγροτική περιοχή, `z = 0.745`) καταλήγει σε `91.67` έναντι συντελεστή κατηγορίας `360`/έτος-όχημα. Τα δύο συμβόλαια αστικής περιοχής που είχαν πράγματι ζημίες, 10002 και 10003, αντίθετα τραβούν το συνδυασμένο κόστος ζημίας τους προς τη δική τους υψηλή εμπειρία (`3039.59` και `3046.88`).
- **Το χρεωθέν ασφάλιστρο** βρίσκεται πάνω από το συνδυασμένο κόστος ζημίας επειδή η `charged_premium` προσθέτει φόρτιση κινδύνου 12% και προσαυξάνει για συντελεστή εξόδων 25%, έναν σταθερό πολλαπλασιαστή `1.12 / 0.75 = 1.493` επί το κόστος ζημίας. Το μέσο χρεωθέν ασφάλιστρο διαμορφώνεται στο `476.61` (αγροτική), `813.04` (προαστιακή) και `1,987.17` (αστική).
- **Προεξοφλημένα αποθέματα:** η `pv_reserve` προεξοφλεί το εκκρεμές απόθεμα υπόθεσης κάθε συμβολαίου (35% της επελθούσας ζημίας) τρία έτη με επιτόκιο 4%, έναν συντελεστή παρούσας αξίας `0.889`. Τα συμβόλαια χωρίς αξιώσεις φέρουν `reserve_pv = 0`· οι δύο αξιώσεις αστικής περιοχής συνεισφέρουν `1,067.95` και `2,226.55`. Συγκεντρωτικά, το βιβλίο διατηρεί προεξοφλημένο απόθεμα `$2,151.56` (αγροτική), `$5,932.52` (προαστιακή) και `$13,364.13` (αστική).
- **Οι υπονοούμενοι δείκτες ζημιών** διαμορφώνονται στο 60.5% (αγροτική), 55.8% (προαστιακή) και 63.6% (αστική) — όλοι άνετα κάτω από 100%, οπότε το φορτισμένο ασφάλιστρο καλύπτει την αναμενόμενη ζημία σε κάθε τμήμα. Το αστικό τμήμα είναι το πιο «καυτό», καθοδηγούμενο από τις δύο μεγάλες προσομοιωμένες ζημίες του· μια πραγματική επανεξέταση συντελεστών θα εξέταζε αν αυτό το σήμα επιμένει σε περισσότερα έτη ατυχημάτων πριν προσαρμοστεί ο χειροκίνητος συντελεστής.

Η υπορουτίνα `blend_premium` επίσης επιδεικνύει το ιδίωμα `OUTARGS` για την επιστροφή πολλαπλών αποτελεσμάτων από μία `CALL` — το συνδυασμένο κόστος ζημίας και ο συντελεστής αξιοπιστίας `z` επιστρέφουν μαζί — αποφεύγοντας ξεχωριστές κλήσεις συναρτήσεων και διατηρώντας τη λογική τιμολόγησης ανά συμβόλαιο συμπαγή και ελέγξιμη.